In [ ]:
import pyproj
pyproj.datadir.set_data_dir("/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj")

In [ ]:
!ls /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj

In [ ]:
import argparse
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
from omegaconf import OmegaConf
from scipy.stats import linregress, rankdata

from config_utils import load_config
from dataloader import make_era5_filename
from preprocessing import (
    Preprocessor,
    SequentialPreprocessor,
    XarrayMinMaxScaler,
    XarrayStandardScaler,
    create_2024_preprocessor,
)

from utils import (
    setup_global_map, setup_north_pole_map, plot_colormesh_on_map,
    plot_global_field, plot_north_pole_field,
    load_rrtm_kernel, interpolate_spatial_field,
    dataset_from_array, ordered_vars, ordered_dataset, filter_by_years,
    preprocessed_feature_target_arrays, empirical_copula_values,
    unpreprocess_feature_values, inverse_transform_without_upsampling, make_kernel_filename,
    load_era5_profile_instant, load_rrtm_kernel_instant
)

from model import SimpleModel

from mpl_toolkits.basemap import Basemap
import xarray as xr
import itertools


In [ ]:
config = load_config("configs/model/sobolev_mar_june_sep_dec_2011_2014.yaml")

# --- load model checkpoint ---
checkpoint_path = Path(config.train.checkpoint_dir) / "model_epoch_43.pt"
checkpoint_data = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

model_weights = checkpoint_data["model_state_dict"]
model_config = checkpoint_data["config"]

model = SimpleModel(model_config)
model.load_state_dict(model_weights)
model.eval()

# --- setup output directory ---
output_dir = Path("scratch_figs_rrtmg_test_2")

if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

# --- load test data ---
year = 2015
month = 12
day = 2
ds = load_era5_profile_instant(year=year, month=month, day=day)
kern = load_rrtm_kernel_instant(year=year, month=month, day=day)
date_string = f"{year}-{month:02d}-{day:02d}"

# --- load preprocessor ---
preprocessor = create_2024_preprocessor()
preprocessor.load(config.preprocess.params_dir)

In [ ]:
plot_global_field(
    ds.tisr.values[0] / 86400, ds.longitude, ds.latitude, f"TISR (ERA5) ({date_string})", output_dir / f"global_tisr_era5_{date_string}.png",
    cmap="viridis", vmin=0, vmax=500, label="$W/m^2$",
    annotation=f"",
)
plot_global_field(
    ds.tsr.values[0] / 86400, ds.longitude, ds.latitude, f"TSR (ERA5) ({date_string})", output_dir / f"global_tsr_era5_{date_string}.png",
    cmap="viridis", vmin=0, vmax=500, label="$W/m^2$",
    annotation=f"",
)

In [ ]:
import itertools
from eval import kernel_date_test

kernel_date_test(
    ds=ds,
    preprocessor=preprocessor,
    model=model,
    date=date_string,
    true_kernel=kern,
    figures_path = output_dir
)

In [ ]:
output_dir_2 = Path("scratch_figs_rrtmg_test_2") / 'features'

if not output_dir_2.exists():
    output_dir_2.mkdir(parents=True, exist_ok=True)

# --- load test data ---
year = 2015
month = 12
day = 2
ds_1 = load_era5_profile_instant(year=year, month=month, day=1)
ds_2 = load_era5_profile_instant(year=year, month=month, day=2)

date_string_1 = f"{year}-{month:02d}-{1:02d}"
date_string_2 = f"{year}-{month:02d}-{2:02d}"

In [ ]:
maxes_1 = np.max(np.abs(ds_1))
maxes_2 = np.max(np.abs(ds_2))
for var_name in ds_1.data_vars:
    v_max = max(maxes_1[var_name].values, maxes_2[var_name].values)
    plot_global_field(
        ds_1[var_name].values[0], ds_1.longitude, ds_1.latitude, f"{var_name} (ERA5) ({date_string_1})", output_dir_2 / f"global_{var_name}_era5_{date_string_1}.png",
        cmap="RdBu_r", vmin=-v_max, vmax=v_max, label="$W/m^2$",
        annotation=f"",
    )
    plot_global_field(
        ds_2[var_name].values[0], ds_1.longitude, ds_1.latitude, f"{var_name} (ERA5) ({date_string_2})", output_dir_2 / f"global_{var_name}_era5_{date_string_2}.png",
        cmap="RdBu_r", vmin=-v_max, vmax=v_max, label="$W/m^2$",
        annotation=f"",
    )

plt.show()

In [ ]:
kern_1 = load_rrtm_kernel_instant(year=year, month=month, day=1)
kern_2 = load_rrtm_kernel_instant(year=year, month=month, day=2)
print(kern_1)
plot_global_field(
    kern_1['TOA'].values[0] / 100, kern_1.longitude, kern_1.latitude, f"kern (ERA5) ({date_string_1})", output_dir_2 / f"kern_era5_{date_string_1}.png",
    cmap="RdBu_r", vmin=-4, vmax=4, label=r"$W/m^2 1\%$",
    annotation=f"",
)
plot_global_field(
    kern_2['TOA'].values[0] / 100, kern_1.longitude, kern_1.latitude, f"kern (ERA5) ({date_string_2})", output_dir_2 / f"kern_era5_{date_string_2}.png",
    cmap="RdBu_r", vmin=-4, vmax=4, label=r"$W/m^2 1\%$",
    annotation=f"",
)

In [ ]:
import os

config = load_config("configs/model/sobolev_mar_june_sep_dec_2011_2014.yaml")

preprocessor = create_2024_preprocessor()
preprocessor.load(config.preprocess.params_dir)

ds_1_p = preprocessor.transform(ds_1)
ds_2_p = preprocessor.transform(ds_2)

maxes_1 = np.max(np.abs(ds_1_p))
maxes_2 = np.max(np.abs(ds_2_p))

os.makedirs(output_dir / 'features_p', exist_ok=True)

for var_name in ds_1_p.data_vars:
    v_max = max(maxes_1[var_name].values, maxes_2[var_name].values)
    plot_global_field(
        ds_1_p[var_name].values[0], ds_1_p.longitude, ds_1_p.latitude, f"{var_name} (ERA5) ({date_string_1}) processed", output_dir / 'features_p' / f"global_{var_name}_era5_{date_string_1}.png",
        cmap="RdBu_r", vmin=-v_max, vmax=v_max, label="$W/m^2$",
        annotation=f"{maxes_1[var_name].values}",
    )
    plot_global_field(
        ds_2_p[var_name].values[0], ds_1_p.longitude, ds_1_p.latitude, f"{var_name} (ERA5) ({date_string_2}) processed", output_dir / 'features_p' / f"global_{var_name}_era5_{date_string_2}.png",
        cmap="RdBu_r", vmin=-v_max, vmax=v_max, label="$W/m^2$",
        annotation=f"{maxes_2[var_name].values}",
    )